<a href="https://colab.research.google.com/github/GARVIT-bit/ai_LAB/blob/main/FOL_FORWARD_chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
class KnowledgeBase:
    def __init__(self):
        # Facts representing atomic sentences
        self.facts = set()
        # Rules stored as (premises, conclusion) tuples.
        # Variables are represented as strings starting with an underscore (e.g., '_x')
        self.rules = []

    def add_fact(self, fact):
        self.facts.add(fact)

    def add_rule(self, premises, conclusion):
        self.rules.append((premises, conclusion))

    def match_and_substitute(self, pattern, fact, env):
        if pattern[0] != fact[0]:
            return None
        if len(pattern) != len(fact):
            return None

        new_env = env.copy()
        for p_arg, f_arg in zip(pattern[1:], fact[1:]):
            if p_arg.startswith('_'):  # It is a variable
                if p_arg in new_env:
                    if new_env[p_arg] != f_arg:
                        return None
                else:
                    new_env[p_arg] = f_arg
            else:  # It is a constant
                if p_arg != f_arg:
                    return None
        return new_env

    def evaluate_premises(self, premises, env, facts_list):
        if not premises:
            yield env
            return

        first, rest = premises[0], premises[1:]
        for fact in facts_list:
            new_env = self.match_and_substitute(first, fact, env)
            if new_env is not None:
                yield from self.evaluate_premises(rest, new_env, facts_list)

    def forward_chain(self, goal):
        print("--- Starting Forward Chaining ---")
        print(f"Initial Facts: {list(self.facts)}\n")

        step = 1
        while True:
            new_facts = set()

            for premises, conclusion in self.rules:
                # Find all valid substitutions for the premises
                for env in self.evaluate_premises(premises, {}, list(self.facts)):
                    # Apply substitution to the conclusion
                    substituted_conclusion = (
                        conclusion[0],
                        *(env.get(arg, arg) for arg in conclusion[1:])
                    )

                    if substituted_conclusion not in self.facts and substituted_conclusion not in new_facts:
                        new_facts.add(substituted_conclusion)
                        premise_str = ' ^ '.join([f"{p[0]}({', '.join(env.get(a, a) for a in p[1:])})" for p in premises])
                        conclusion_str = f"{substituted_conclusion[0]}({', '.join(substituted_conclusion[1:])})"
                        print(f"Step {step}: Inferred {conclusion_str} from {premise_str}")
                        step += 1

            if not new_facts:
                print("\nNo new facts can be inferred.")
                break

            self.facts.update(new_facts)

            if goal in self.facts:
                print(f"\nGoal {goal[0]}({', '.join(goal[1:])}) successfully PROVED!")
                return True

        print(f"\nGoal {goal[0]}({', '.join(goal[1:])}) could NOT be proved.")
        return False

# Initialize Knowledge Base
kb = KnowledgeBase()

# 1. Base Facts (from slides)
# American(Robert)
kb.add_fact(('American', 'Robert'))
# Enemy(A, America)
kb.add_fact(('Enemy', 'A', 'America'))
# Owns(A, T1)
kb.add_fact(('Owns', 'A', 'T1'))
# Missile(T1)
kb.add_fact(('Missile', 'T1'))

# 2. Rules (with variables prefixed with '_')
# Missile(x) => Weapon(x)
kb.add_rule([('Missile', '_x')], ('Weapon', '_x'))

# Enemy(x, America) => Hostile(x)
kb.add_rule([('Enemy', '_x', 'America')], ('Hostile', '_x'))

# Missile(x) ^ Owns(A, x) => Sells(Robert, x, A)
kb.add_rule([('Missile', '_x'), ('Owns', 'A', '_x')], ('Sells', 'Robert', '_x', 'A'))

# American(p) ^ Weapon(q) ^ Sells(p, q, r) ^ Hostile(r) => Criminal(p)
kb.add_rule(
    [
        ('American', '_p'),
        ('Weapon', '_q'),
        ('Sells', '_p', '_q', '_r'),
        ('Hostile', '_r')
    ],
    ('Criminal', '_p')
)

# Prove that Robert is Criminal
goal_state = ('Criminal', 'Robert')
kb.forward_chain(goal_state)

--- Starting Forward Chaining ---
Initial Facts: [('American', 'Robert'), ('Missile', 'T1'), ('Enemy', 'A', 'America'), ('Owns', 'A', 'T1')]

Step 1: Inferred Weapon(T1) from Missile(T1)
Step 2: Inferred Hostile(A) from Enemy(A, America)
Step 3: Inferred Sells(Robert, T1, A) from Missile(T1) ^ Owns(A, T1)
Step 4: Inferred Criminal(Robert) from American(Robert) ^ Weapon(T1) ^ Sells(Robert, T1, A) ^ Hostile(A)

Goal Criminal(Robert) successfully PROVED!


True